# Iris Classifier using Vertex AI

### Dataset

This tutorial uses R.A. Fisher's Iris dataset, a small and popular dataset for machine learning experiments. Each instance has four numerical features, which are different measurements of a flower, and a target label that
categorizes the flower into: **Iris setosa**, **Iris versicolour** and **Iris virginica**.

This tutorial uses [a version of the Iris dataset available in the
scikit-learn library](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html#sklearn.datasets.load_iris).

### Install Vertex AI SDK for Python and other required packages



In [1]:
# Vertex SDK for Python
! pip3 install --upgrade --quiet  google-cloud-aiplatform

### Set Google Cloud project information

In [2]:
PROJECT_ID = "project-e1f2f75a-c8a7-4a76-a66"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}

### Create a Cloud Storage bucket

In [3]:
BUCKET_URI = f"gs://mlops-course-dproject-e1f2f75a-c8a7-4a76-a66-unique"  # @param {type:"string"}
MODEL_ARTIFACT_DIR="iris_classifier/model"

In [4]:
#! gsutil mb -l {LOCATION} -p {PROJECT_ID} {BUCKET_URI}
! gcloud storage buckets create {BUCKET_URI} --location="US"

Creating gs://mlops-course-dproject-e1f2f75a-c8a7-4a76-a66-unique/...
ERROR: (gcloud.storage.buckets.create) HTTPError 409: Your previous request to create the named bucket succeeded and you already own it.


### Initialize Vertex AI SDK for Python

In [5]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET_URI)

### Import the required libraries

In [6]:
import os
import sys

## Simple Decision Tree model
Build a Decision Tree model on iris data

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pandas.plotting import parallel_coordinates
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn import metrics
from datetime import datetime

import pickle
import joblib

# Using Raw Data

In [8]:
raw_data = pd.read_csv('data/raw/iris.csv')
raw_data.head(5)

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [9]:
raw_data.shape

(150, 5)

In [10]:
def training_pipeline(data, test_size_split):
    train, test = train_test_split(data, test_size = test_size_split, stratify = data['species'], random_state = 42)
    X_train = train[['sepal_length','sepal_width','petal_length','petal_width']]
    y_train = train.species
    X_test = test[['sepal_length','sepal_width','petal_length','petal_width']]
    y_test = test.species

    model = DecisionTreeClassifier(max_depth = 3, random_state = 1)
    model.fit(X_train, y_train)

    return X_test, y_test, model

In [11]:
X_raw_test, y_raw_test, mod_dt = training_pipeline(raw_data, 0.4)

In [12]:
prediction = mod_dt.predict(X_raw_test)
print('The accuracy of the Decision Tree is',"{:.3f}".format(metrics.accuracy_score(prediction,y_raw_test)))

The accuracy of the Decision Tree is 0.983


In [13]:
joblib.dump(mod_dt, "artifacts/raw_data_model.joblib")

['artifacts/raw_data_model.joblib']

### Upload model artifacts and custom code to Cloud Storage

In [14]:
# !gsutil cp artifacts/model.joblib {BUCKET_URI}/{MODEL_ARTIFACT_DIR}/
!gcloud storage cp artifacts/raw_data_model.joblib {BUCKET_URI}/{MODEL_ARTIFACT_DIR}/raw_data_model.joblib

Copying file://artifacts/raw_data_model.joblib to gs://mlops-course-dproject-e1f2f75a-c8a7-4a76-a66-unique/iris_classifier/model/raw_data_model.joblib
  Completed files 1/1 | 2.5kiB/2.5kiB                                          


## Model Inferencing

In [15]:
!gcloud storage cp {BUCKET_URI}/{MODEL_ARTIFACT_DIR}/raw_data_model.joblib artifacts/downloaded_raw_data_model.joblib

Copying gs://mlops-course-dproject-e1f2f75a-c8a7-4a76-a66-unique/iris_classifier/model/raw_data_model.joblib to file://artifacts/downloaded_raw_data_model.joblib
  Completed files 1/1 | 2.5kiB/2.5kiB                                          


In [16]:
dw_raw_data_model = joblib.load("artifacts/downloaded_raw_data_model.joblib")
dw_pred = dw_raw_data_model.predict(X_raw_test)
print('Accuracy from the downloaded model is',"{:.3f}".format(metrics.accuracy_score(dw_pred,y_raw_test)))

Accuracy from the downloaded model is 0.983


### Task 02: Store Training data in GCS

In [17]:
!gcloud storage cp data/v1/data.csv {BUCKET_URI}/"iris_classifier/data"/"v1"/data.csv
!gcloud storage cp data/v2/data.csv {BUCKET_URI}/"iris_classifier/data"/"v2"/data.csv

Copying file://data/v1/data.csv to gs://mlops-course-dproject-e1f2f75a-c8a7-4a76-a66-unique/iris_classifier/data/v1/data.csv
  Completed files 1/1 | 2.5kiB/2.5kiB                                          
Copying file://data/v2/data.csv to gs://mlops-course-dproject-e1f2f75a-c8a7-4a76-a66-unique/iris_classifier/data/v2/data.csv
  Completed files 1/1 | 1.3kiB/1.3kiB                                          


### Task 03: Execute the IRIS Training Pipeline

In [36]:
# Fetch the data from GCS
data_v1 = pd.read_csv(f"{BUCKET_URI}/iris_classifier/data/v1/data.csv")
data_v2 = pd.read_csv(f"{BUCKET_URI}/iris_classifier/data/v2/data.csv")

In [37]:
print(f"Version 01 data shape: {data_v1.shape}")
print(f"Version 02 data shape: {data_v2.shape}")

Version 01 data shape: (101, 5)
Version 02 data shape: (49, 5)


In [38]:
X_test_01, y_test_01, model_01 = training_pipeline(data_v1, 0.4)

In [39]:
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
ARTIFACT_FOLDER = f"artifacts/{TIMESTAMP}"
os.makedirs(ARTIFACT_FOLDER, exist_ok=True)

In [40]:
local_model_path = os.path.join(ARTIFACT_FOLDER, "model_v1.joblib")
joblib.dump(model_01, local_model_path)

# Storing artifacts in GCS Bucket
gcs_model_path = f"{BUCKET_URI}/{MODEL_ARTIFACT_DIR}/v1/{TIMESTAMP}/model_v1.joblib"
!gcloud storage cp {local_model_path} {gcs_model_path}

I1010 16:20:57.277251   41419 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


Copying file://artifacts/20261010_162056/model_v1.joblib to gs://mlops-course-dproject-e1f2f75a-c8a7-4a76-a66-unique/iris_classifier/model/v1/20261010_162056/model_v1.joblib
  Completed files 1/1 | 2.3kiB/2.3kiB                                          


### Task 04: Run Inference on Evaluation Set

In [41]:
# Download model file from GCS to local directory
!gcloud storage cp {gcs_model_path} {ARTIFACT_FOLDER}/model_v1_gcs.joblib

I1010 16:20:59.392259   41419 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


Copying gs://mlops-course-dproject-e1f2f75a-c8a7-4a76-a66-unique/iris_classifier/model/v1/20261010_162056/model_v1.joblib to file://artifacts/20261010_162056/model_v1_gcs.joblib
  Completed files 1/1 | 2.3kiB/2.3kiB                                          


In [42]:
model_01_gcs = joblib.load(f"{ARTIFACT_FOLDER}/model_v1_gcs.joblib")

pred_01 = model_01_gcs.predict(X_test_01)
print('Accuracy from downloaded gcs model_01 is',"{:.3f}".format(metrics.accuracy_score(pred_01, y_test_01)))

Accuracy from downloaded gcs model_01 is 0.951


### Task 05: Run Training & Inference Twice

In [43]:
def run_training_inferencing(data, version):
    TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
    ARTIFACT_FOLDER = f"artifacts/{TIMESTAMP}"
    os.makedirs(ARTIFACT_FOLDER, exist_ok=True)
    
    X_test, y_test, model = training_pipeline(data, 0.4)
    
    local_model_path = os.path.join(ARTIFACT_FOLDER, f"model_{version}.joblib")
    joblib.dump(model, local_model_path)
    
    # Storing artifacts in GCS Bucket
    gcs_model_path = f"{BUCKET_URI}/{MODEL_ARTIFACT_DIR}/{version}/{TIMESTAMP}/model_{version}.joblib"
    !gcloud storage cp {local_model_path} {gcs_model_path}

    # Inferencing from the second run
    !gcloud storage cp {gcs_model_path} {ARTIFACT_FOLDER}/model_{version}_gcs.joblib
    
    model_gcs = joblib.load(f"{ARTIFACT_FOLDER}/model_{version}_gcs.joblib")
    
    pred = model_gcs.predict(X_test)
    print(f'Accuracy from downloaded gcs model_{version} is',"{:.3f}".format(metrics.accuracy_score(pred, y_test)))

In [44]:
run_training_inferencing(data_v1, "v1")

I1010 16:21:01.396209   41419 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


Copying file://artifacts/20261010_162101/model_v1.joblib to gs://mlops-course-dproject-e1f2f75a-c8a7-4a76-a66-unique/iris_classifier/model/v1/20261010_162101/model_v1.joblib
  Completed files 1/1 | 2.3kiB/2.3kiB                                          


I1010 16:21:04.438628   41419 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


Copying gs://mlops-course-dproject-e1f2f75a-c8a7-4a76-a66-unique/iris_classifier/model/v1/20261010_162101/model_v1.joblib to file://artifacts/20261010_162101/model_v1_gcs.joblib
  Completed files 1/1 | 2.3kiB/2.3kiB                                          
Accuracy from downloaded gcs model_v1 is 0.951


### Task 6: Multiple Data Versions

In [45]:
# Run the whole pipeline with version 02 data 
run_training_inferencing(data_v2, "v2")

I1010 16:21:06.464831   41419 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


Copying file://artifacts/20261010_162106/model_v2.joblib to gs://mlops-course-dproject-e1f2f75a-c8a7-4a76-a66-unique/iris_classifier/model/v2/20261010_162106/model_v2.joblib
  Completed files 1/1 | 2.2kiB/2.2kiB                                          


I1010 16:21:08.562741   41419 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


Copying gs://mlops-course-dproject-e1f2f75a-c8a7-4a76-a66-unique/iris_classifier/model/v2/20261010_162106/model_v2.joblib to file://artifacts/20261010_162106/model_v2_gcs.joblib
  Completed files 1/1 | 2.2kiB/2.2kiB                                          
Accuracy from downloaded gcs model_v2 is 1.000


In [46]:
# Run the whole pipeline with version 02 data for second time
run_training_inferencing(data_v2, "v2")

I1010 16:21:10.633979   41419 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


Copying file://artifacts/20261010_162110/model_v2.joblib to gs://mlops-course-dproject-e1f2f75a-c8a7-4a76-a66-unique/iris_classifier/model/v2/20261010_162110/model_v2.joblib
  Completed files 1/1 | 2.2kiB/2.2kiB                                          


I1010 16:21:13.020042   41419 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


Copying gs://mlops-course-dproject-e1f2f75a-c8a7-4a76-a66-unique/iris_classifier/model/v2/20261010_162110/model_v2.joblib to file://artifacts/20261010_162110/model_v2_gcs.joblib
  Completed files 1/1 | 2.2kiB/2.2kiB                                          
Accuracy from downloaded gcs model_v2 is 1.000
